In [1]:
import json
import ollama

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

In [2]:
server_params = StdioServerParameters(
    command="python3",
    args=["accounts_server.py"]
)

print("✅ Accounts MCP Server configured")

✅ Accounts MCP Server configured


In [3]:
async def list_account_tools():

    async with stdio_client(server_params) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            tools = await session.list_tools()

            return tools

In [6]:
tools = await list_account_tools()

print(tools)

meta=None nextCursor=None tools=[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None), Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotati

In [5]:
import subprocess

result = subprocess.run(
    ["python3", "accounts_server.py"],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)

print("\nSTDERR:")
print(result.stderr)

STDOUT:


STDERR:
Traceback (most recent call last):
  File "/Users/prateekchoudhary/Downloads/agents-main/6_mcp/accounts_server.py", line 2, in <module>
    from accounts import Account
  File "/Users/prateekchoudhary/Downloads/agents-main/6_mcp/accounts.py", line 5, in <module>
    from market import get_share_price
  File "/Users/prateekchoudhary/Downloads/agents-main/6_mcp/market.py", line 1, in <module>
    from polygon import RESTClient
ModuleNotFoundError: No module named 'polygon'



In [7]:
async def get_balance(name):

    async with stdio_client(server_params) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            result = await session.call_tool(
                "get_balance",
                {
                    "name": name
                }
            )

            return result

In [8]:
balance = await get_balance("ed")

print(balance)

meta=None content=[TextContent(type='text', text='10000.0', annotations=None, meta=None)] structuredContent={'result': 10000.0} isError=False


In [9]:
async def read_account_resource(name):

    async with stdio_client(server_params) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            result = await session.read_resource(
                f"accounts://accounts_server/{name}"
            )

            return result.contents[0].text

In [11]:
account_data = await read_account_resource("ed")

print(account_data)

{"name": "ed", "balance": 10000.0, "strategy": "", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-06-19 23:28:33", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}


In [12]:
def ask_ollama(prompt):

    response = ollama.chat(
        model="qwen2.5:7b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]

In [13]:
prompt = f"""
You are a financial advisor.

Analyze this account:

{account_data}

Provide:

1. Current Balance
2. Portfolio Summary
3. Risk Assessment
4. Investment Strategy Suggestions
5. Next Recommended Action
"""

result = ask_ollama(prompt)

print(result)

Certainly! Let's break down the provided account information and provide a comprehensive analysis.

### 1. Current Balance

The current balance of the account is **$10,000.00**.

### 2. Portfolio Summary

- **Holdings**: The portfolio summary indicates that there are no current holdings as `holdings` is an empty object `{}`.
- **Transactions**: There are also no recorded transactions in the `transactions` array, which means no trades or activities have been recorded since the account was created.
- **Portfolio Value Time Series**:
  - As of June 19, 2026 (the first data point), the portfolio value is $10,000.00.

### 3. Risk Assessment

Given the current state of the portfolio—no active holdings or transactions—the risk profile appears to be minimal. However, it’s important to note that:
- **Market Risks**: While currently there are no investments, any potential market fluctuations could impact the value if/when assets are acquired.
- **Liquidity Risk**: The account has liquidity as ca

In [14]:
async def analyze_account(name):

    account_data = await read_account_resource(name)

    prompt = f"""
You are a professional financial advisor.

Analyze this account:

{account_data}

Provide:

1. Current Financial Position
2. Portfolio Summary
3. Risk Assessment
4. Investment Opportunities
5. Recommended Next Actions
"""

    return ask_ollama(prompt)

In [15]:
await analyze_account("ed")

'Certainly! Let\'s analyze the provided account details for an individual named Ed.\n\n### 1. Current Financial Position\n\n- **Account Name:** ed\n- **Current Balance:** $10,000.00\n- **Strategy:** Not specified (empty string)\n- **Holdings:** None (holding dictionary is empty)\n- **Transactions:** None (transactions list is empty)\n- **Portfolio Value Time Series:**\n  - On both timestamps provided ("2026-06-19 23:28:33" and "2026-06-19 23:31:08"), the portfolio value was $10,000.00.\n- **Total Portfolio Value:** $10,000.00\n- **Total Profit/Loss:** $0.00\n\n### 2. Portfolio Summary\n\n- The account currently has no active investments or holdings.\n- There have been no transactions recorded as of the provided data.\n\n### 3. Risk Assessment\n\nGiven that Ed\'s portfolio is empty, there are currently zero financial risks associated with his account. However, it is important to note that without any investment, Ed’s purchasing power and potential for growth in wealth remain at their in

In [16]:
async def buy_stock(name, symbol, quantity, rationale):

    async with stdio_client(server_params) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            result = await session.call_tool(
                "buy_shares",
                {
                    "name": name,
                    "symbol": symbol,
                    "quantity": quantity,
                    "rationale": rationale
                }
            )

            return result

In [17]:
result = await buy_stock(
    name="ed",
    symbol="AAPL",
    quantity=5,
    rationale="Long-term growth investment"
)

print(result)

meta=None content=[TextContent(type='text', text='Error executing tool buy_shares: 1 validation error for buy_sharesOutput\nresult\n  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value=\'Completed. Latest detail...ss": 89.26000000000022}\', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.11/v/float_parsing', annotations=None, meta=None)] structuredContent=None isError=True


In [18]:
account_data = await read_account_resource("ed")
print(account_data)

{"name": "ed", "balance": 9629.26, "strategy": "", "holdings": {"AAPL": 5}, "transactions": [{"symbol": "AAPL", "quantity": 5, "price": 74.148, "timestamp": "2026-06-19 23:49:42", "rationale": "Long-term growth investment"}], "portfolio_value_time_series": [["2026-06-19 23:28:33", 10000.0], ["2026-06-19 23:31:08", 10000.0], ["2026-06-19 23:49:42", 10089.26], ["2026-06-19 23:50:17", 9749.26]], "total_portfolio_value": 9749.26, "total_profit_loss": -250.73999999999978}


In [19]:
async def get_holdings(name):

    async with stdio_client(server_params) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            result = await session.call_tool(
                "get_holdings",
                {
                    "name": name
                }
            )

            return result

In [20]:
holdings = await get_holdings("ed")

print(holdings)

meta=None content=[TextContent(type='text', text='{\n  "AAPL": 5\n}', annotations=None, meta=None)] structuredContent={'AAPL': 5} isError=False


In [21]:
async def sell_stock(name, symbol, quantity, rationale):

    async with stdio_client(server_params) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            result = await session.call_tool(
                "sell_shares",
                {
                    "name": name,
                    "symbol": symbol,
                    "quantity": quantity,
                    "rationale": rationale
                }
            )

            return result

In [22]:
result = await sell_stock(
    name="ed",
    symbol="AAPL",
    quantity=2,
    rationale="Partial profit booking"
)

print(result)

meta=None content=[TextContent(type='text', text='Error executing tool sell_shares: 1 validation error for sell_sharesOutput\nresult\n  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value=\'Completed. Latest detail...ss": -75.0679999999993}\', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.11/v/float_parsing', annotations=None, meta=None)] structuredContent=None isError=True


In [23]:
account_data = await read_account_resource("ed")

print(account_data)

{"name": "ed", "balance": 9792.932, "strategy": "", "holdings": {"AAPL": 3}, "transactions": [{"symbol": "AAPL", "quantity": 5, "price": 74.148, "timestamp": "2026-06-19 23:49:42", "rationale": "Long-term growth investment"}, {"symbol": "AAPL", "quantity": -2, "price": 81.836, "timestamp": "2026-06-19 23:52:29", "rationale": "Partial profit booking"}], "portfolio_value_time_series": [["2026-06-19 23:28:33", 10000.0], ["2026-06-19 23:31:08", 10000.0], ["2026-06-19 23:49:42", 10089.26], ["2026-06-19 23:50:17", 9749.26], ["2026-06-19 23:52:29", 9924.932], ["2026-06-19 23:52:55", 9843.932]], "total_portfolio_value": 9843.932, "total_profit_loss": -156.0679999999993}
